# Level 3: Multi-Qubit Systems and the Magic of Entanglement
## Q# Edition

In this notebook, you will:

1. Work with **multi-qubit systems** in Q#
2. Understand and create **entanglement**
3. Build **Bell States** using `CNOT`
4. Verify entanglement through correlated measurements

In [ ]:
import qsharp
from qsharp_widgets import Circuit
print("Q# ready!")

---
## 1. Multi-Qubit Allocation in Q#

Q# makes it easy to work with multiple qubits using `use`:

In [ ]:
%%qsharp

// Allocate two qubits individually
operation TwoQubitsIndividual() : (Result, Result) {
    use (q1, q2) = (Qubit(), Qubit());
    let r1 = M(q1);
    let r2 = M(q2);
    Reset(q1);
    Reset(q2);
    return (r1, r2);
}

// Or allocate an array of qubits
operation ThreeQubitsArray() : Result[] {
    use qs = Qubit[3];
    let results = MeasureEachZ(qs);
    ResetAll(qs);
    return results;
}

In [ ]:
print("Two individual qubits:", qsharp.eval("TwoQubitsIndividual()"))
print("Three qubits as array:", qsharp.eval("ThreeQubitsArray()"))
print("All zeros — no gates applied!")

---
## 2. The CNOT Gate

In Q#, the CNOT gate is called with `CNOT(control, target)`:

In [ ]:
%%qsharp

operation DemoCNOT() : Unit {
    // Case 1: Control = |0>, Target = |0>
    use (c, t) = (Qubit(), Qubit());
    CNOT(c, t);
    Message("CNOT with control=|0>:");
    DumpMachine();  // Should be |00>
    ResetAll([c, t]);

    // Case 2: Control = |1>, Target = |0>
    use (c2, t2) = (Qubit(), Qubit());
    X(c2);  // Set control to |1>
    CNOT(c2, t2);
    Message("\nCNOT with control=|1>:");
    DumpMachine();  // Should be |11>
    ResetAll([c2, t2]);
}

In [ ]:
qsharp.eval("DemoCNOT()")

---
## 3. Creating a Bell State

The Bell State recipe: H on the control qubit, then CNOT.

This creates: (1/sqrt(2))(|00> + |11>)

In [ ]:
%%qsharp

operation CreateBellState() : (Result, Result) {
    use (q1, q2) = (Qubit(), Qubit());

    // Create the Bell State
    H(q1);          // Put q1 in superposition
    CNOT(q1, q2);   // Entangle q1 and q2

    // Inspect the state
    Message("Bell State (before measurement):");
    DumpMachine();

    // Measure both qubits
    let r1 = M(q1);
    let r2 = M(q2);

    ResetAll([q1, q2]);
    return (r1, r2);
}

In [ ]:
# Run once to see the state
result = qsharp.eval("CreateBellState()")
print(f"\nSingle measurement: {result}")
print("Both qubits always agree!")

In [ ]:
# Visualize the Bell State circuit
Circuit(qsharp.circuit("CreateBellState()"))

### Verifying Entanglement with Statistics

Let's run the Bell State measurement many times to prove the correlation:

In [ ]:
%%qsharp

operation BellStatistics(shots : Int) : (Int, Int, Int, Int) {
    mutable count00 = 0;
    mutable count01 = 0;
    mutable count10 = 0;
    mutable count11 = 0;

    for _ in 0..shots - 1 {
        use (q1, q2) = (Qubit(), Qubit());
        H(q1);
        CNOT(q1, q2);

        let r1 = M(q1);
        let r2 = M(q2);

        if r1 == Zero and r2 == Zero { set count00 += 1; }
        elif r1 == Zero and r2 == One { set count01 += 1; }
        elif r1 == One and r2 == Zero { set count10 += 1; }
        else { set count11 += 1; }

        ResetAll([q1, q2]);
    }

    return (count00, count01, count10, count11);
}

In [ ]:
result = qsharp.eval("BellStatistics(10000)")
print(f"Results over 10,000 measurements:")
print(f"  |00>: {result[0]}")
print(f"  |01>: {result[1]}")
print(f"  |10>: {result[2]}")
print(f"  |11>: {result[3]}")
print(f"\nOnly |00> and |11> appear — perfect entanglement!")

---
## 4. GHZ State — Three-Qubit Entanglement

The GHZ state extends entanglement to three qubits: (1/sqrt(2))(|000> + |111>)

In [ ]:
%%qsharp

operation CreateGHZ() : Result[] {
    use qs = Qubit[3];

    // GHZ recipe: H on first, CNOT chain
    H(qs[0]);
    CNOT(qs[0], qs[1]);
    CNOT(qs[0], qs[2]);

    Message("GHZ State:");
    DumpMachine();

    let results = MeasureEachZ(qs);
    ResetAll(qs);
    return results;
}

In [ ]:
# Run GHZ multiple times
for i in range(5):
    result = qsharp.eval("CreateGHZ()")
    print(f"Run {i+1}: {result}")

print("\nAll three qubits always agree — either all Zero or all One!")

In [ ]:
# Visualize the GHZ circuit
Circuit(qsharp.circuit("CreateGHZ()"))

---
## 5. Q# Convenience: ApplyToEach

Q# has convenient operations for working with qubit arrays:

In [ ]:
%%qsharp

// A cleaner way to create the GHZ state using ApplyToEach
operation GHZClean(n : Int) : Result[] {
    use qs = Qubit[n];

    H(qs[0]);
    // Apply CNOT from first qubit to each subsequent qubit
    ApplyToEach(CNOT(qs[0], _), qs[1...]);

    let results = MeasureEachZ(qs);
    ResetAll(qs);
    return results;
}

In [ ]:
# Create a 5-qubit GHZ state!
for i in range(5):
    result = qsharp.eval("GHZClean(5)")
    print(f"5-qubit GHZ run {i+1}: {result}")

print("\nAll five qubits are perfectly correlated!")

---
## 6. Key Takeaways

| Concept | Q# Syntax | Description |
|---------|-----------|-------------|
| **Multi-qubit** | `use (q1, q2) = (Qubit(), Qubit());` | Allocate individually |
| **Qubit array** | `use qs = Qubit[n];` | Allocate an array |
| **CNOT** | `CNOT(control, target)` | Controlled NOT gate |
| **Bell State** | `H(q1); CNOT(q1, q2);` | Simplest entangled state |
| **ApplyToEach** | `ApplyToEach(Op, qs)` | Apply operation to each qubit |
| **MeasureEachZ** | `MeasureEachZ(qs)` | Measure all qubits |

---
## 7. Challenges

1. **Anti-correlated Bell**: Modify the Bell State to produce only |01> and |10> (hint: add an X gate).
2. **N-qubit GHZ**: Run `GHZClean` with 10 qubits. Do all 10 still agree?
3. **Bell with DumpRegister**: Use `DumpRegister` to inspect individual qubits of the Bell state. What do you see?

In [ ]:
%%qsharp

// Your challenge code here!

---
**Next up: [Level 4 — Quantum Protocols: Teleportation & Superdense Coding](../Level_04_Quantum_Protocols/Level_04_QSharp.ipynb)**